In [3]:
# %% Cell: THE HONEST RESNET (Safe Features + Weighted Loss)
import pandas as pd
import numpy as np
import os
import gc
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans

# 1. SETUP & DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🏆 INITIALIZING HONEST RESNET on {device}...")

# 2. LOAD DATA
if os.path.exists('/kaggle/input'):
    search_path = '/kaggle/input'
    for root, dirs, files in os.walk(search_path):
        if 'train_v2.csv' in files:
            DATA_PATH = root; break
else:
    DATA_PATH = '../data/raw'

print("   📂 Loading Raw Data...")
train_df = pd.read_csv(os.path.join(DATA_PATH, 'train_v2.csv'))
test_df = pd.read_csv(os.path.join(DATA_PATH, 'test_v2.csv'))

# Sorting is critical for time-series lags
train_df = train_df.sort_values(['date_id', 'minute_id']).reset_index(drop=True)
test_df = test_df.sort_values(['date_id', 'minute_id']).reset_index(drop=True)

# 3. FEATURE ENGINEERING (The "Safe Mode" Logic)
print("   ⚙️ Engineering Safe Features (No Leakage)...")
target_cols = ['target_short', 'target_medium', 'target_long']
ignore = ['id', 'date_id', 'minute_id'] + target_cols
base_features = [c for c in train_df.columns if c not in ignore]
vip_features = ['feature_19', 'feature_5', 'feature_27', 'feature_2', 'feature_13']

def engineer_safe(df):
    df_eng = df.copy()
    
    # A. Lags (Standard)
    for col in base_features:
        for lag in [1, 2, 3, 5]:
            df_eng[f'{col}_lag{lag}'] = df_eng.groupby('date_id')[col].shift(lag)
            
    # B. Rolling Means (Safe Context)
    for w in [5, 10, 20]:
        for col in vip_features:
            df_eng[f'{col}_mean_{w}'] = df_eng.groupby('date_id')[col].transform(lambda x: x.rolling(w).mean())
            df_eng[f'{col}_std_{w}'] = df_eng.groupby('date_id')[col].transform(lambda x: x.rolling(w).std())
            
    # C. ROLLING RANKS (The Fix!)
    # We rank against the last 60 minutes, NOT the whole day.
    for col in vip_features:
        df_eng[f'{col}_rank'] = df_eng.groupby('date_id')[col].transform(
            lambda x: x.rolling(window=60, min_periods=10).rank(pct=True)
        ) - 0.5
        
    return df_eng

train_eng = engineer_safe(train_df)
y = train_eng[target_cols].values
test_eng = engineer_safe(test_df)

del train_df, test_df
gc.collect()

# 4. PREPARE, CLIP & SCALE
final_cols = [c for c in train_eng.columns if c not in ignore]

# --- 🛑 FIX: CLIP BEFORE CREATING NUMPY ARRAYS 🛑 ---
print("   ✂️ Applying 1% / 99% Hard Clipping (Host Strategy)...")
for col in final_cols:
    # Calculate limits on Train
    lower = train_eng[col].quantile(0.01)
    upper = train_eng[col].quantile(0.99)
    
    # Apply to Train AND Test
    train_eng[col] = train_eng[col].clip(lower, upper)
    test_eng[col]  = test_eng[col].clip(lower, upper)

# NOW create the numpy arrays using the clipped data
X = train_eng[final_cols].values
X_test = test_eng[final_cols].values

# Force-Clean NaNs (Safety Check)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)
y = np.nan_to_num(y, nan=0.0)

print("   ⚖️ Robust Scaling...")
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# 5. CLUSTERING
print("   🧩 Generating Market States...")
# Fit only on Train to avoid test-distribution leakage
kmeans = KMeans(n_clusters=7, random_state=42, n_init=10)
kmeans.fit(X_scaled[::10]) 
train_clusters = kmeans.predict(X_scaled)
test_clusters = kmeans.predict(X_test_scaled)

# --- MODEL DEFINITIONS ---

# 1. Weighted Loss (The Metric Hack)
class WeightedMAELoss(nn.Module):
    def __init__(self, weights=[0.5, 0.3, 0.2]):
        super().__init__()
        self.weights = torch.tensor(weights).to(device)
        
    def forward(self, inputs, targets):
        abs_err = torch.abs(inputs - targets)
        weighted_err = torch.sum(abs_err * self.weights, dim=1)
        return torch.mean(weighted_err)

# 2. Residual Block
class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout)
        )
        
    def forward(self, x):
        return x + self.block(x)

# 3. SniperResNet Architecture
class SniperResNet(nn.Module):
    def __init__(self, num_inputs, hidden_dim=512, output_dim=3, dropout=0.25):
        super().__init__()
        self.embedding = nn.Embedding(7, 16)
        self.entry = nn.Sequential(
            nn.Linear(num_inputs, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout)
        )
        self.res_blocks = nn.Sequential(
            ResidualBlock(hidden_dim, dropout),
            ResidualBlock(hidden_dim, dropout),
            ResidualBlock(hidden_dim, dropout)
        )
        self.merge = nn.Sequential(
            nn.Linear(hidden_dim + 16, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
        
    def forward(self, x_num, x_cat):
        emb = self.embedding(x_cat)
        x = self.entry(x_num)
        x = self.res_blocks(x) 
        concat = torch.cat([x, emb], dim=1)
        return self.merge(concat)

# --- TRAINING LOOP WITH OOF SAVING ---

# CONFIG
SEEDS = [42, 43, 44, 101, 777]
N_FOLDS = 5
EPOCHS = 25
BATCH_SIZE = 2048
LEARNING_RATE = 5e-4 
TARGET_SCALE = 100.0 

print(f"🚀 Training SniperResNet on {device}...")

X_num_tensor = torch.FloatTensor(X_scaled).to(device)
X_test_num_tensor = torch.FloatTensor(X_test_scaled).to(device)
cluster_train_tensor = torch.LongTensor(train_clusters).to(device)
cluster_test_tensor = torch.LongTensor(test_clusters).to(device)
y_tensor = torch.FloatTensor(y * TARGET_SCALE).to(device)

# ARRAYS TO HOLD PREDICTIONS
ensemble_preds = np.zeros((len(X_test_scaled), 3))
oof_preds = np.zeros((len(X_scaled), 3)) # <--- NEW: Holds Validation Predictions

for seed in SEEDS:
    print(f"\n🌱 Seed {seed}...")
    kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    seed_test_preds = np.zeros((len(X_test_scaled), 3))
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_num_tensor, y_tensor)):
        
        train_ds = TensorDataset(X_num_tensor[train_idx], cluster_train_tensor[train_idx], y_tensor[train_idx])
        val_ds = TensorDataset(X_num_tensor[val_idx], cluster_train_tensor[val_idx], y_tensor[val_idx])
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False)
        
        model = SniperResNet(num_inputs=X_num_tensor.shape[1]).to(device)
        criterion = WeightedMAELoss(weights=[0.5, 0.3, 0.2]) 
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LEARNING_RATE, 
                                                  steps_per_epoch=len(train_loader), epochs=EPOCHS)
        
        best_wmae = float('inf')
        best_weights = copy.deepcopy(model.state_dict())
        
        for epoch in range(EPOCHS):
            model.train()
            for bx_num, bx_cat, by in train_loader:
                optimizer.zero_grad()
                out = model(bx_num, bx_cat)
                loss = criterion(out, by)
                loss.backward()
                optimizer.step()
                scheduler.step()
            
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for bx_num, bx_cat, by in val_loader:
                    out = model(bx_num, bx_cat)
                    val_loss += criterion(out, by).item() * bx_num.size(0)
            
            avg_wmae = val_loss / len(val_ds)
            
            if avg_wmae < best_wmae:
                best_wmae = avg_wmae
                best_weights = copy.deepcopy(model.state_dict())
        
        # --- INFERENCE ON VALIDATION (OOF) & TEST ---
        model.load_state_dict(best_weights)
        model.eval()
        with torch.no_grad():
            # 1. Generate OOF for this fold
            # We must iterate specifically over the val_loader to keep order
            fold_oof = []
            for bx_num, bx_cat, _ in val_loader:
                fold_oof.append(model(bx_num, bx_cat).cpu().numpy())
            
            # Save to global OOF array (Divide by scale to get real units)
            # We average over seeds, so we add 1/len(SEEDS) portion
            oof_preds[val_idx] += (np.concatenate(fold_oof) / TARGET_SCALE) / len(SEEDS)

            # 2. Generate Test Preds
            fold_p = []
            test_loader = DataLoader(TensorDataset(X_test_num_tensor, cluster_test_tensor), batch_size=4096)
            for bx_num, bx_cat in test_loader:
                fold_p.append(model(bx_num, bx_cat).cpu().numpy())
            seed_test_preds += np.concatenate(fold_p) / N_FOLDS
        
        del model, optimizer; torch.cuda.empty_cache()
    
    ensemble_preds += seed_test_preds / len(SEEDS)
    print(f"   ✅ Seed {seed} Done.")

# --- SAVING OOF ---
# We need the IDs for the OOF file to match them later
oof_df = pd.DataFrame({'id': train_eng['id'].values}) # Ensure train_eng is still in memory or reload it
oof_df['target_short'] = oof_preds[:, 0]
oof_df['target_medium'] = oof_preds[:, 1]
oof_df['target_long'] = oof_preds[:, 2]
oof_df.to_csv('../submissions/oof_resnet_honest.csv', index=False)
print("🚀 Saved 'oof_resnet_honest.csv' (Validation Predictions)")

# --- SAVING SUBMISSION ---
final_preds = ensemble_preds / TARGET_SCALE
sub_df = pd.read_csv(os.path.join(DATA_PATH, 'test_v2.csv'), usecols=['id'])
sub_df['target_short'] = final_preds[:, 0]
sub_df['target_medium'] = final_preds[:, 1]
sub_df['target_long'] = final_preds[:, 2]

# Zero-Mean Centering
for c in ['target_short', 'target_medium', 'target_long']:
    sub_df[c] = sub_df[c] - sub_df[c].mean()

sub_df.to_csv('../submissions/submission_resnet_honest.csv', index=False)
print("🚀 Saved 'submission_resnet_honest.csv'.")

🏆 INITIALIZING HONEST RESNET on cpu...
   📂 Loading Raw Data...
   ⚙️ Engineering Safe Features (No Leakage)...


/var/folders/1w/qc85tcfd69b0fy564_d2zdmh0000gn/T/ipykernel_8880/3990640011.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_eng[f'{col}_lag{lag}'] = df_eng.groupby('date_id')[col].shift(lag)
/var/folders/1w/qc85tcfd69b0fy564_d2zdmh0000gn/T/ipykernel_8880/3990640011.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_eng[f'{col}_lag{lag}'] = df_eng.groupby('date_id')[col].shift(lag)
/var/folders/1w/qc85tcfd69b0fy564_d2zdmh0000gn/T/ipykernel_8880/3990640011.py:49: PerformanceWarning: DataFrame is highly fragmented.  Thi

   ✂️ Applying 1% / 99% Hard Clipping (Host Strategy)...
   ⚖️ Robust Scaling...
   🧩 Generating Market States...
🚀 Training SniperResNet on cpu...

🌱 Seed 42...
   ✅ Seed 42 Done.

🌱 Seed 43...
   ✅ Seed 43 Done.

🌱 Seed 44...
   ✅ Seed 44 Done.

🌱 Seed 101...
   ✅ Seed 101 Done.

🌱 Seed 777...
   ✅ Seed 777 Done.
🚀 Saved 'oof_resnet_honest.csv' (Validation Predictions)
🚀 Saved 'submission_resnet_honest.csv'.
